# Set Up

## Mount Google Drive

Ignore if not using Google Collab:

In [20]:
from google.colab import drive

# mount google drive
drive.mount('/content/drive')
%cd /content/drive/My Drive
!git clone https://github.com/FranciscoLozCoding/cooling_with_code.git
%cd cooling_with_code
!git pull

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive
fatal: destination path 'cooling_with_code' already exists and is not an empty directory.
/content/drive/My Drive/cooling_with_code
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 479 bytes | 0 bytes/s, done.
From https://github.com/FranciscoLozCoding/cooling_with_code
   480b39d..7596bad  main       -> origin/main
Updating 480b39d..7596bad
Fast-forward
 tools/explainer.py | 6 +++++-
 1 file changed, 5 insertions(+), 1 deletion(-)
fatal: cannot exec '.git/hooks/post-merge': Permission denied


## Import Libraries

In [21]:
%pip install shap xgboost

In [22]:
# Supress Warnings
import warnings
warnings.filterwarnings('ignore')

#data science
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.feature_selection import RFECV
import shap
from sklearn.model_selection import GridSearchCV

#other
import os
import pickle

#custom tools
from tools.environment import VALID_SPLIT, RANDOM_STATE
from tools.preprocess import load_and_preprocess_data

## Import Datasets

In [37]:
#load in dataset
csv_path = 'data/train/300m_buffer_dataset.csv'

# no split
X, Y, scaler = load_and_preprocess_data(csv_path, split=False)

# split
x_train, x_valid, y_train, y_valid, scaler = load_and_preprocess_data(
    csv_path, split=True,
    test_size=VALID_SPLIT, random_state=RANDOM_STATE)

Loading data from data/train/300m_buffer_dataset.csv
Loading data from data/train/300m_buffer_dataset.csv


# XgBoost Model

This notebook is for creating a XgBoost model using our 300m Buffer Dataset. We will apply what we learned in notebooks 4-6 (eg; [4](/04_EDA.ipynb), [5](/05_preprocessing.ipynb), [6](06_increasing_buffer_zone.ipynb)) to improve the model. For details on the train/test dataset refer to our past notebooks:
- [01_dataset_generation](/01_dataset_generation.ipynb)
- [02_more_dataset_generation](/02_more_dataset_generation.ipynb)

## Feature Selection

First, we will do feature selection. We now have interaction terms, so the dataset is filled with a lot of features that need to be dropped.

In [38]:
# Ensure x_train is a DataFrame
x_train_df = pd.DataFrame(x_train, columns=X.columns)

In [39]:
# Initialize XGBoost model
xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.05,
                         max_depth=12, random_state=RANDOM_STATE, objective="reg:squarederror")

# Recursive Feature Elimination with Cross-Validation
rfe = RFECV(estimator=xgb_model, step=1, cv=5, scoring="r2", n_jobs=-1)

# Fit to training data
rfe.fit(x_train_df, y_train)

# Get selected features
selected_features = x_train_df.loc[:, rfe.support_]

# Get names of selected features
selected_feature_names = x_train_df.columns[rfe.support_]

# Print selected feature names
print("Selected Features:", list(selected_feature_names))

Selected Features: ['NDVI', 'SI', 'NDMI', 'NPCRI', 'Coastal_Aerosol', 'Building_Count', 'Total_Building_Area_m2', 'Building_Height', 'Building_Construction_Year', 'Ground_Elevation', 'Traffic_Volume', 'Building_Wind_X', 'Building_Wind_Y', 'Elevation_Wind_X', 'Elevation_Wind_Y', 'BldgHeight_Count', 'BuildingDensity_NDVI', 'Traffic_NDVI', 'Temp_BuildingDensity', 'Humidity_NDVI', 'Humidity_NDMI', 'Traffic_NDBI', 'Traffic_BuildingDensity', 'BuildingAge_Temp']


In [40]:
# save the selected features so we don't have to run again
selected_features=['NDVI', 'SI', 'NDMI', 'NPCRI', 'Coastal_Aerosol', 'Building_Count',
                   'Total_Building_Area_m2', 'Building_Height', 'Building_Construction_Year',
                   'Ground_Elevation', 'Traffic_Volume', 'Building_Wind_X', 'Building_Wind_Y',
                   'Elevation_Wind_X', 'Elevation_Wind_Y', 'BldgHeight_Count', 'BuildingDensity_NDVI',
                   'Traffic_NDVI', 'Temp_BuildingDensity', 'Humidity_NDVI', 'Humidity_NDMI',
                   'Traffic_NDBI', 'Traffic_BuildingDensity', 'BuildingAge_Temp']

Here we see the features that were selected. These features maximized the cross-validation score. Now let's select them for our training/validation set.

In [41]:
# select features
x_train_selected = pd.DataFrame(x_train, columns=x_train_df.columns)[selected_features]
x_valid_selected = pd.DataFrame(x_valid, columns=x_train_df.columns)[selected_features]

## Hyperparameter Tuning

Now we will tune the XGBRegressor hyperparemeters using `GridSearchCV`.

In [42]:
# Define the base XGBoost model
xgb_model = XGBRegressor(objective="reg:squarederror", random_state=RANDOM_STATE)

# Define the grid of hyperparameters to search
param_grid = {
    "n_estimators": [100, 200, 300],  # Number of trees
    "learning_rate": [0.05, 0.1, 0.2],  # Step size
    "max_depth": [5, 7, 12],  # Tree depth
    "min_child_weight": [1, 3, 5],  # Minimum weight sum for leaf node
    "subsample": [0.8, 1.0],  # Fraction of samples per tree
    "colsample_bytree": [0.8, 1.0],  # Fraction of features per tree
    "gamma": [0, 0.1, 0.2],  # Minimum loss reduction for a split
    "reg_lambda": [None, 1, 10],  # L2 regularization (None = No L2)
    "reg_alpha": [None, 0, 1]  # L1 regularization (None = No L1)
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring="r2",  # Optimize for R-squared score
    cv=5,  # 5-fold cross-validation
    verbose=2,
    n_jobs=-1  # Use all available CPU cores
)

# Fit grid search to the **selected features only**
grid_search.fit(x_train_selected, y_train)

# Print the best hyperparameters and best score
print("Best Hyperparameters:", grid_search.best_params_)
print("Best R-squared Score:", grid_search.best_score_)

Fitting 5 folds for each of 8748 candidates, totalling 43740 fits
Best Hyperparameters: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 12, 'min_child_weight': 5, 'n_estimators': 300, 'reg_alpha': None, 'reg_lambda': None, 'subsample': 0.8}
Best R-squared Score: 0.9488482807899755


Here we see the hyper params for the best model and the R-squared it got. Let's retrieve the best model.

In [43]:
# best model, skip if model was already generated
model_pkl = 'models/300m_XGBRegressor/300m_XGBRegressor_model.pkl'
if not os.path.exists(model_pkl):
  best_xgb_model = grid_search.best_estimator_
else:
  with open(model_pkl, 'rb') as f:
    best_xgb_model = pickle.load(f)

## In-Sample Evaluation

In [44]:
# Make predictions on the training data
insample_predictions = best_xgb_model.predict(x_train_selected)

# calculate R-squared score for in-sample predictions
print(f"In-Sample Evaluation:")

insample_r2 = r2_score(y_train, insample_predictions)
print(f"  300m buffer zone R-squared: {insample_r2}")

In-Sample Evaluation:
  300m buffer zone R-squared: 0.9977363451435304


## Out-Sample Evaluation

In [45]:
# Make predictions on the validation data
outsample_predictions = best_xgb_model.predict(x_valid_selected)

# calculate R-squared score for in-sample predictions
print(f"Out-Sample Evaluation:")

outsample_r2 = r2_score(y_valid, outsample_predictions)
print(f"  300m buffer zone R-squared: {outsample_r2}")

Out-Sample Evaluation:
  300m buffer zone R-squared: 0.9567974751946091


# Challenge Submission

>TODO

# Save The Model

In [46]:
# Save the model and scaler to files
with open('300m_XGBRegressor_model.pkl', 'wb') as f:
    pickle.dump(best_xgb_model, f)
with open('300m_standard_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)